# Notebook 04 · Feature Engineering
**Input :**
```
F:\mmd\data\cleaned\sessions.pkl
F:\mmd\data\cleaned\reviews_clean.parquet
F:\mmd\data\cleaned\meta_clean.parquet
F:\mmd\data\vocab\item2idx.json
```
**Output:**
```
F:\mmd\data\features\
    item2vec_corpus.txt          ← sentences cho gensim Word2Vec
    gru4rec_train.pkl            ← list of (input_seq, target_item)
    gru4rec_val.pkl
    gru4rec_test.pkl
    item_side_features.parquet   ← price_bin, category_id, avg_rating (optional)
    feature_config.json          ← N_ITEMS, N_USERS, MAX_SEQ_LEN, PAD_IDX…
```
---
### Pipeline
```
sessions.pkl
    │
    ├── Item2Vec corpus ──► mỗi train_seq là 1 "sentence" (space-separated item_idx)
    │
    └── GRU4Rec dataset
            │
            ├── sliding window trên train_seq
            │     input = seq[i:i+window], target = seq[i+window]
            │     (dùng toàn bộ prefix thay vì fixed window)
            ├── val:  input = train_seq, target = val_item
            └── test: input = train_seq + [val_item], target = test_item
```

## 0 · Imports & config

In [ ]:
import gc, json, pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import psutil
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

def ram():
    v = psutil.virtual_memory()
    return f"RAM {v.used/1e9:.1f}/{v.total/1e9:.1f} GB ({v.percent:.0f}%)"

ROOT_DIR     = Path(r"F:\amazon_data")
CLEANED_DIR  = ROOT_DIR / "data" / "cleaned"
VOCAB_DIR    = ROOT_DIR / "data" / "vocab"
FEATURE_DIR  = ROOT_DIR / "data" / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

MAX_SEQ_LEN  = 50     
                      
PAD_IDX      = 0      
                     
RANDOM_SEED  = 42
np.random.seed(RANDOM_SEED)

print(f"Feature dir : {FEATURE_DIR}")
print(ram())

Feature dir : F:\amazon_data\data\features
RAM 3.9/8.4 GB (46%)


## 1 · Load data

In [ ]:

with open(CLEANED_DIR / "sessions.pkl", "rb") as f:
    sessions = pickle.load(f)
print(f"Sessions loaded : {len(sessions):,} users")
print(f"Columns         : {sessions.columns.tolist()}")


with open(VOCAB_DIR / "item2idx.json") as f:
    item2idx = json.load(f)
with open(VOCAB_DIR / "idx2item.json") as f:
    idx2item = json.load(f)

N_ITEMS_ORIGINAL = len(item2idx)


df_meta = pd.read_parquet(CLEANED_DIR / "meta_clean.parquet")

print(f"Vocab size      : {N_ITEMS_ORIGINAL:,} items")
print(f"Meta rows       : {len(df_meta):,}")
print(ram())

Sessions loaded : 2,712,338 users
Columns         : ['user_idx', 'train_seq', 'val_item', 'test_item', 'seq_len']
Vocab size      : 667,277 items
Meta rows       : 667,277
RAM 5.3/8.4 GB (63%)


## 2 · Shift item indices (PAD_IDX = 0)
Để dùng `padding_idx=0` trong `nn.Embedding`, ta shift tất cả item index lên 1.  
→ `item_idx_shifted = original_item_idx + 1`  
→ `N_ITEMS = N_ITEMS_ORIGINAL + 1` (slot 0 dành cho PAD)

In [ ]:
def shift_seq(seq: list) -> list:
    return [i + 1 for i in seq]   

sessions["train_seq"] = sessions["train_seq"].apply(shift_seq)
sessions["val_item"]  = sessions["val_item"].apply(lambda x: x + 1)
sessions["test_item"] = sessions["test_item"].apply(lambda x: x + 1)

N_ITEMS = N_ITEMS_ORIGINAL + 1   
print(f"N_ITEMS (with PAD) : {N_ITEMS:,}")
print(f"PAD_IDX            : {PAD_IDX}")
print(f"Item idx range     : 1 … {N_ITEMS-1}")

N_ITEMS (with PAD) : 667,278
PAD_IDX            : 0
Item idx range     : 1 … 667277


## 3 · Item2Vec corpus
Mỗi `train_seq` là 1 "sentence", item index là "word".  
Gensim `Word2Vec` nhận đầu vào là list of list of strings.

In [ ]:


corpus = [list(map(str, seq)) for seq in sessions["train_seq"] if len(seq) >= 2]
print(f"Corpus sentences : {len(corpus):,}")
print(f"Sample sentence  : {corpus[0][:10]} ...")


corpus_path = FEATURE_DIR / "item2vec_corpus.txt"
with open(corpus_path, "w") as f:
    for sent in corpus:
        f.write(" ".join(sent) + "\n")
print(f"Corpus saved → {corpus_path}")
print(f"File size    : {corpus_path.stat().st_size / 1e6:.1f} MB")


with open(FEATURE_DIR / "item2vec_corpus.pkl", "wb") as f:
    pickle.dump(corpus, f, protocol=4)
print(f"Corpus pkl saved → {FEATURE_DIR / 'item2vec_corpus.pkl'}")

Corpus sentences : 2,712,338
Sample sentence  : ['500522', '151774', '419574', '69738', '216510', '215622', '7305', '295', '5135', '126615'] ...
Corpus saved → F:\amazon_data\data\features\item2vec_corpus.txt
File size    : 115.4 MB
Corpus pkl saved → F:\amazon_data\data\features\item2vec_corpus.pkl


## 4 · GRU4Rec dataset
Tạo 3 tập dữ liệu cho GRU4Rec theo **3 bước**:

### Bước 1 — Train samples (all-prefix strategy)
Với mỗi 	rain_seq = [i1, i2, ..., iT], sinh ra T-1 samples:
- [i1] → target i2
- [i1, i2] → target i3
- ...
- [i1, ..., i_{T-1}] → target iT

Mỗi input được **left-pad** đến MAX_SEQ_LEN.  
Để tránh tràn RAM, samples được ghi xuống ổ đĩa từng chunk (~2M dòng), sau đó gộp thành 1 file gru4rec_train.parquet bằng ParquetWriter.

### Bước 2 — Val & Test samples
- **Val** : input = pad(train_seq) → target = al_item
- **Test**: input = pad(train_seq + [val_item]) → target = 	est_item

### Output
| File | Định dạng | Nội dung |
|---|---|---|
| gru4rec_train.parquet | Parquet (snappy) | (input_seq, target) cho train |
| gru4rec_val.pkl | Pickle | (input_seq, target) cho val |
| gru4rec_test.pkl | Pickle | (input_seq, target) cho test |

In [5]:
gc.collect()

7

In [ ]:
import gc
import pickle
import psutil
import shutil
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

print("BƯỚC 1/3: Xử lý tập Train cuốn chiếu (Tránh tràn RAM)...")


TEMP_TRAIN_DIR = FEATURE_DIR / "_temp_train"
TEMP_TRAIN_DIR.mkdir(parents=True, exist_ok=True)


for f in TEMP_TRAIN_DIR.glob("*.parquet"): 
    f.unlink()

def pad_seq(seq: list, max_len: int, pad_val: int = 0) -> list:
    if len(seq) >= max_len:
        return seq[-max_len:]
    return [pad_val] * (max_len - len(seq)) + seq

inputs_buffer, targets_buffer = [], []
chunk_id = 0
total_train = 0
BUFFER_SIZE = 2_000_000 

pbar = tqdm(sessions["train_seq"], desc="Train Samples")
for seq in pbar:
    if len(seq) < 2: 
        continue
    for t in range(1, len(seq)):
        inputs_buffer.append(pad_seq(seq[:t], MAX_SEQ_LEN, PAD_IDX))
        targets_buffer.append(seq[t])
        total_train += 1
        
     
        if len(inputs_buffer) >= BUFFER_SIZE:
            df_chunk = pd.DataFrame({"input_seq": inputs_buffer, "target": targets_buffer})
            df_chunk["target"] = df_chunk["target"].astype(np.int32) 
            
            df_chunk.to_parquet(TEMP_TRAIN_DIR / f"chunk_{chunk_id:04d}.parquet", engine="pyarrow", index=False)
            
            chunk_id += 1
            inputs_buffer, targets_buffer = [], []
            del df_chunk
            gc.collect()
            pbar.set_postfix({"Samples": f"{total_train:,}", "RAM": ram()})


if inputs_buffer: 
    df_chunk = pd.DataFrame({"input_seq": inputs_buffer, "target": targets_buffer})
    df_chunk["target"] = df_chunk["target"].astype(np.int32)
    df_chunk.to_parquet(TEMP_TRAIN_DIR / f"chunk_{chunk_id:04d}.parquet", engine="pyarrow", index=False)
    inputs_buffer, targets_buffer = [], []
    del df_chunk
    gc.collect()

print("BƯỚC 2/3: Gộp tập Train thành 1 file duy nhất bằng ParquetWriter...")


train_output_path = FEATURE_DIR / "gru4rec_train.parquet" 

chunk_files = sorted(TEMP_TRAIN_DIR.glob("chunk_*.parquet"))
writer = None
for chunk_file in tqdm(chunk_files, desc="Merging Train"):
    table = pq.read_table(chunk_file)
    if writer is None:
        writer = pq.ParquetWriter(train_output_path, table.schema, compression='snappy')
    writer.write_table(table)
    del table
    gc.collect()

if writer:
    writer.close()


shutil.rmtree(TEMP_TRAIN_DIR)

print("BƯỚC 3/3: Xử lý và lưu tập Val & Test (Giữ nguyên định dạng .pkl cũ)...")

val_samples, test_samples = [], []

for row in tqdm(sessions.itertuples(), total=len(sessions), desc="Val & Test"):
   
    val_samples.append((pad_seq(list(row.train_seq), MAX_SEQ_LEN, PAD_IDX), row.val_item))
 
    test_samples.append((pad_seq(list(row.train_seq) + [row.val_item], MAX_SEQ_LEN, PAD_IDX), row.test_item))


with open(FEATURE_DIR / "gru4rec_val.pkl", "wb") as f:
    pickle.dump(val_samples, f, protocol=4)

with open(FEATURE_DIR / "gru4rec_test.pkl", "wb") as f:
    pickle.dump(test_samples, f, protocol=4)

del val_samples, test_samples
gc.collect()

print("\n=== HOÀN TẤT ===")
print(f"Đã lưu Train: {train_output_path}")
print(f"Đã lưu Val  : {FEATURE_DIR / 'gru4rec_val.pkl'}")
print(f"Đã lưu Test : {FEATURE_DIR / 'gru4rec_test.pkl'}")
print(ram())

BƯỚC 1/3: Xử lý tập Train cuốn chiếu (Tránh tràn RAM)...


Train Samples:   0%|          | 0/2712338 [00:00<?, ?it/s]

BƯỚC 2/3: Gộp tập Train thành 1 file duy nhất bằng ParquetWriter...


Merging Train:   0%|          | 0/9 [00:00<?, ?it/s]

BƯỚC 3/3: Xử lý và lưu tập Val & Test (Giữ nguyên định dạng .pkl cũ)...


Val & Test:   0%|          | 0/2712338 [00:00<?, ?it/s]


=== HOÀN TẤT ===
Đã lưu Train: F:\amazon_data\data\features\gru4rec_train.parquet
Đã lưu Val  : F:\amazon_data\data\features\gru4rec_val.pkl
Đã lưu Test : F:\amazon_data\data\features\gru4rec_test.pkl
RAM 5.8/8.4 GB (69%)


## 5 · Item side features
Encode `price_bin`, `store` (brand), `average_rating` → dùng optional cho GRU4Rec augmented.

In [ ]:
import re

def parse_price(s):
    if pd.isna(s) or str(s).strip() in ("", "None"):
        return float("nan")
    nums = re.findall(r"[\d]+(?:\.[\d]+)?", str(s))
    if not nums: return float("nan")
    return round(sum(float(n) for n in nums) / len(nums), 2)

if "price_usd" not in df_meta.columns:
    df_meta["price_usd"] = df_meta["price"].apply(parse_price)


price_edges = [0, 10, 25, 50, 100, 200, float("inf")]
df_meta["price_bin_id"] = pd.cut(
    df_meta["price_usd"],
    bins=price_edges,
    labels=False,        
    include_lowest=True
).fillna(6).astype(int)   
N_PRICE_BINS = 7

TOP_K_STORES = 500
store_counts = df_meta["store"].value_counts()
top_stores   = set(store_counts.head(TOP_K_STORES).index)
store2id     = {s: i+1 for i, s in enumerate(store_counts.head(TOP_K_STORES).index)}
df_meta["store_id"] = df_meta["store"].map(store2id).fillna(0).astype(int)
N_STORES = TOP_K_STORES + 1   


df_meta["avg_rating_norm"] = (
    (df_meta["average_rating"].fillna(0) - 1) / 4
).clip(0, 1).astype("float32")


import numpy as np
df_meta["rating_count_log"] = np.log1p(
    df_meta["rating_number"].fillna(0)
).astype("float32")


side_cols = ["item_idx", "parent_asin", "price_bin_id",
             "store_id", "avg_rating_norm", "rating_count_log"]
df_side = df_meta[side_cols].copy()


df_side["item_idx"] = df_side["item_idx"] + 1


pad_row = pd.DataFrame([{"item_idx": 0, "parent_asin": "__PAD__",
                          "price_bin_id": 0, "store_id": 0,
                          "avg_rating_norm": 0.0, "rating_count_log": 0.0}])
df_side = pd.concat([pad_row, df_side], ignore_index=True)

df_side.to_parquet(FEATURE_DIR / "item_side_features.parquet", index=False)
print(f"Side features shape : {df_side.shape}")
print(df_side.head(3).to_string())

Side features shape : (667278, 6)
   item_idx parent_asin  price_bin_id  store_id  avg_rating_norm  rating_count_log
0         0     __PAD__             0         0            0.000          0.000000
1    329101  B0BNZ8Q7YT             1         0            0.850          4.912655
2    127678  B00KKU8HTG             6         0            0.825          5.123964


## 6 · Save feature config

In [ ]:
feature_config = {
    "N_ITEMS"       : N_ITEMS,          
    "N_USERS"       : int(sessions["user_idx"].nunique()),
    "N_PRICE_BINS"  : N_PRICE_BINS,
    "N_STORES"      : N_STORES,
    "MAX_SEQ_LEN"   : MAX_SEQ_LEN,
    "PAD_IDX"       : PAD_IDX,
    "RANDOM_SEED"   : RANDOM_SEED,
    "TS_UNIT"       : "ms",

    "item2vec_dim"  : 128,
    "item2vec_window": 5,
    "gru_hidden"    : 256,
    "gru_layers"    : 1,
    "embed_dim"     : 128,
}

config_path = FEATURE_DIR / "feature_config.json"
with open(config_path, "w") as f:
    json.dump(feature_config, f, indent=2)

print("Feature config saved:")
for k, v in feature_config.items():
    print(f"  {k:<20}: {v}")

Feature config saved:
  N_ITEMS             : 667278
  N_USERS             : 2712338
  N_PRICE_BINS        : 7
  N_STORES            : 501
  MAX_SEQ_LEN         : 50
  PAD_IDX             : 0
  RANDOM_SEED         : 42
  TS_UNIT             : ms
  item2vec_dim        : 128
  item2vec_window     : 5
  gru_hidden          : 256
  gru_layers          : 1
  embed_dim           : 128


## 7 · Sanity check

In [ ]:

with open(FEATURE_DIR / "gru4rec_val.pkl", "rb") as f:
    val_check = pickle.load(f)

inp_sample, tgt_sample = val_check[0]
print(f"Val sample[0]:")
print(f"  input shape : {len(inp_sample)} tokens")
print(f"  input (last 10): {inp_sample[-10:]}")
print(f"  target       : {tgt_sample}")
print(f"  target range : 1 … {N_ITEMS-1} (PAD=0)")
assert 1 <= tgt_sample <= N_ITEMS - 1, "Target out of range!"
assert len(inp_sample) == MAX_SEQ_LEN, "Input length mismatch!"
assert inp_sample.count(0) >= 0, "PAD check ok"

print("\n" + "="*50)
print("  FEATURE ENGINEERING COMPLETE")
print("="*50)
print(f"  {FEATURE_DIR}")
for p in sorted(FEATURE_DIR.glob("*")):
    print(f"    {p.name:<35} {p.stat().st_size/1e6:>8.1f} MB")
print(ram())

Val sample[0]:
  input shape : 50 tokens
  input (last 10): [169938, 7495, 54616, 74600, 1860, 1217, 18777, 16502, 55111, 3635]
  target       : 357571
  target range : 1 … 667277 (PAD=0)

  FEATURE ENGINEERING COMPLETE
  F:\amazon_data\data\features
    feature_config.json                      0.0 MB
    gru4rec_test.pkl                       334.2 MB
    gru4rec_train.parquet                  423.6 MB
    gru4rec_val.pkl                        329.6 MB
    item2vec_corpus.pkl                    162.6 MB
    item2vec_corpus.txt                    115.4 MB
    item_side_features.parquet              12.6 MB
RAM 7.9/8.4 GB (94%)


---
## Notes

| Quyết định | Lý do |
|---|---|
| PAD_IDX = 0, shift item +1 | Convention chuẩn cho `nn.Embedding(padding_idx=0)` |
| All-prefix training | Tận dụng tối đa mỗi sequence, phổ biến hơn fixed-window |
| Left-pad thay vì right-pad | GRU4Rec đọc từ trái sang phải, padding bên trái → hidden state cuối = context thực |
| MAX_SEQ_LEN = 50 | Cân bằng RAM (7.9 GB) vs độ dài cần thiết |
| Side features optional | Item2Vec không cần; GRU4Rec base cũng không; chỉ dùng nếu augment |

**→ Notebook 05a:** Train Item2Vec  
**→ Notebook 05b:** Train GRU4Rec